# Hệ thống RAG cho Luật Giao thông Việt Nam (2008)

Notebook này xây dựng một hệ thống RAG (Retrieval-Augmented Generation) hoàn toàn offline sử dụng LangChain, ChromaDB và mô hình ngôn ngữ lớn Qwen 2.5.

In [ ]:
# CÀI ĐẶT THƯ VIỆN & THIẾT LẬP ĐƯỜNG DẪN
!pip install -q torch>=2.0.0 transformers>=4.40.0 accelerate>=0.30.0 huggingface-hub>=0.23.0
!pip install -q sentence-transformers>=2.7.0 langchain-core>=0.2.0 langchain-chroma>=0.2.0 langchain-huggingface bitsandbytes chromadb>=0.5.0

import os
import json
import torch
from google.colab import drive

# Kết nối Google Drive
drive.mount('/content/drive', force_remount=True)
BASE_PATH = "/content/drive/MyDrive/NLP"
DATASET_DIR = os.path.join(BASE_PATH, "data/datasets")

# Thư mục chứa Vector DB mới
VECTOR_DB_DIR = os.path.join(BASE_PATH, "vector_db_parent_child")
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

PARENT_JSON_FILE = os.path.join(DATASET_DIR, "parents_traffic_law.json")
CHILD_JSON_FILE = os.path.join(DATASET_DIR, "children_traffic_law.json")

# Token
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"

Mounted at /content/drive


## 2. Load dữ liệu đã xử lý
Chúng ta sẽ tải các đoạn văn bản (chunks) từ file JSON đã được tiền xử lý trước đó vào định dạng `Document` của LangChain.

In [ ]:
# ==============================================================================
# 2. XÂY DỰNG CUSTOM PARENT-CHILD RETRIEVER
# ==============================================================================
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableLambda

print("[*] Đang tải dữ liệu JSON...")
# 1. Tải Parents
with open(PARENT_JSON_FILE, "r", encoding="utf-8") as f:
    parents_data = json.load(f)
print(f"[+] Đã tải {len(parents_data)} Parent Documents.")

# 2. Tải Children
with open(CHILD_JSON_FILE, "r", encoding="utf-8") as f:
    children_data = json.load(f)
print(f"[+] Đã tải {len(children_data)} Child Chunks.")

# 3. Khởi tạo Document Store cho Parent (Lưu trên RAM)
docstore = InMemoryStore()
parent_docs_kv = [
    (p["doc_id"], Document(page_content=p["content"], metadata=p["metadata"]))
    for p in parents_data
]
docstore.mset(parent_docs_kv)

# 4. Khởi tạo Vector Store cho Child (Lưu xuống Disk)
print("[*] Khởi tạo ChromaDB cho Child Chunks...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = Chroma(
    collection_name="child_chunks_collection",
    embedding_function=embedding_model,
    persist_directory=VECTOR_DB_DIR
)

# Thêm dữ liệu vào VectorStore (chỉ chạy lần đầu, nếu DB rỗng)
if vectorstore._collection.count() == 0:
    print("[*] Vector DB trống. Đang Indexing Child Chunks...")
    child_docs = [Document(page_content=c["content"], metadata=c["metadata"]) for c in children_data]
    vectorstore.add_documents(child_docs)
    print("[+] Hoàn tất Indexing!")
else:
    print(f"[+] Vector DB đã có sẵn {vectorstore._collection.count()} chunks. Load thành công!")

def retrieve_parent_docs(query: str, k: int = 3):
    """
    1. Tìm 'k' chunks con khớp với câu hỏi nhất.
    2. Lấy ra 'parent_id' của chúng (lọc trùng lặp bằng set).
    3. Trả về toàn bộ nội dung của các Parent Documents gốc.
    """
    child_docs = vectorstore.similarity_search(query, k=k)

    parent_ids = list(set([doc.metadata.get("parent_id") for doc in child_docs if "parent_id" in doc.metadata]))

    parent_docs = []
    if parent_ids:
        docs = docstore.mget(parent_ids)
        parent_docs = [d for d in docs if d is not None]

    return parent_docs

# Bọc hàm Python thành một Runnable chuẩn của LangChain
custom_retriever = RunnableLambda(lambda query: retrieve_parent_docs(query, k=3))

[*] Đang tải dữ liệu JSON...
[+] Đã tải 176 Parent Documents.
[+] Đã tải 707 Child Chunks.
[*] Khởi tạo ChromaDB cho Child Chunks...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[*] Vector DB trống. Đang Indexing Child Chunks...
[+] Hoàn tất Indexing!
